In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from concept_abstraction.training import train_ppo_model, SimpleQEstimator
from concept_abstraction.selection import greedy_selection_supervised
from concept_abstraction.env_utils import *
from concept_abstraction.utils import *
import sys 
import argparse
import secrets
import numpy as np 
import random 
import time 
from collections import Counter
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score


/usr0/home/naveenr/.local/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
is_jupyter = 'ipykernel' in sys.modules

In [5]:
if is_jupyter: 
    seed        = 42
    num_concepts_selected = 20
else:
    parser = argparse.ArgumentParser()
    parser.add_argument('--seed', help='Random Seed', type=int, default=42)
    parser.add_argument('--environment_string', help='Which environment to create', type=str, default="tree")
    parser.add_argument('--training_timesteps', help='Number of training timesteps', type=int, default=10000)
    parser.add_argument('--num_concepts_selected', help='Number of concepts selected by greedy or random',type=int, default=0)
    parser.add_argument('--selection_function', help='When selecting, use q_value, policy, or transition?', type=str, default="policy")
    parser.add_argument('--target_abstraction', help='Value for the target abstraction with human performance', type=float, default=0.05)
    parser.add_argument('--human_accuracy_by_concept', nargs='*', type=float, default=None)
    parser.add_argument('--cbm_accuracy_by_concept', help="What is the accuracy of AI per concept?", nargs='*', type=float, default=None)
    parser.add_argument('--human_reliance_by_concept', help="How much does AI rely on human intervention?",  nargs='*', type=float, default=None)
    parser.add_argument('--reward_error', help="How much to perturb the reward by?", type=float, default=0)
    parser.add_argument('--out_folder', help='Which folder', type=str, default="exploration")

    args = parser.parse_args()

    seed = args.seed
    environment_string = args.environment_string
    training_timesteps = args.training_timesteps 
    selection_function = args.selection_function
    num_concepts_selected = args.num_concepts_selected
    human_accuracy_by_concept = args.human_accuracy_by_concept
    human_reliance_by_concept = args.human_reliance_by_concept
    target_abstraction = args.target_abstraction
    cbm_accuracy_by_concept = args.cbm_accuracy_by_concept
    reward_error = args.reward_error
    out_folder = args.out_folder

save_name = secrets.token_hex(4)  

In [6]:
results = {}
results['parameters'] = {'seed'      : seed,
        'num_concepts_selected': num_concepts_selected,
}
print("Parameters {}".format(results['parameters']))

Parameters {'seed': 42, 'environment_string': 'cart_pole_binary', 'training_timesteps': 10000, 'selection_function': 'policy', 'num_concepts_selected': 20, 'human_accuracy_by_concept': None, 'human_reliance_by_concept': None, 'target_abstraction': 0.05, 'cbm_accuracy_by_concept': [0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1], 'reward_error': 0}


In [7]:
np.random.seed(seed)
random.seed(seed)

In [8]:
dataset = json.load(open("../../data/cub/preprocessed.json"))

## Concept Selection

In [9]:
train_X = np.array([row['attributes'] for row in dataset['train']])
test_X = np.array([row['attributes'] for row in dataset['test']])

In [10]:
all_rows_train = set([''.join([str(int(j)) for j in row]) for row in train_X])
all_rows_test = set([''.join([str(int(j)) for j in row]) for row in test_X])

In [11]:
def get_performance(selected_concepts,accuracy_by_concept):
    train_X = np.array([row['attributes'] for row in dataset['train']])
    test_X = np.array([row['attributes'] for row in dataset['test']])

    flip_probs = 1 - np.array(accuracy_by_concept)
    rand_vals = np.random.rand(*train_X.shape)
    flip_mask = rand_vals < flip_probs  # True means flip
    train_X = np.where(flip_mask, 1 - train_X, train_X)
    train_X = train_X[:,selected_concepts]

    flip_probs = 1 - np.array(accuracy_by_concept)
    rand_vals = np.random.rand(*test_X.shape)
    flip_mask = rand_vals < flip_probs  # True means flip
    test_X = np.where(flip_mask, 1 - test_X, test_X)
    test_X = test_X[:,selected_concepts]


    train_Y = np.array([row['label'] for row in dataset['train']])
    test_Y = np.array([row['label'] for row in dataset['test']])

    mlp = MLPClassifier(
        hidden_layer_sizes=(50),
        activation='relu',
        solver='adam',
        max_iter=1000,  # increase if needed
        random_state=0,
        alpha=1e-3,  # instead of 0.0001,
        early_stopping=True, validation_fraction=0.1, n_iter_no_change=20
    )

    # Train the model
    mlp.fit(train_X, train_Y)

    # Predict on the test set
    y_pred = mlp.predict(test_X)

    # Compute accuracy
    acc = accuracy_score(test_Y, y_pred)
    return acc

In [12]:
train_X = np.array([row['attributes'] for row in dataset['train']])
test_X = np.array([row['attributes'] for row in dataset['test']])
train_Y = np.array([row['label'] for row in dataset['train']])
test_Y = np.array([row['label'] for row in dataset['test']])


In [33]:
random_concept_list = [np.random.choice(list(range(train_X.shape[1])),k,replace=False) for k in range(1,num_concepts_selected)]
random_average_reward = [get_performance(c,np.ones(312)) for c in random_concept_list]

results['random_selection'] = {
    'concepts': random_concept_list, 
    'values': random_average_reward,
}

In [41]:
greedy_concept_list = greedy_selection_supervised(train_X,train_Y,num_concepts_selected)
greedy_reward = [get_performance(c,np.ones(312)) for c in greedy_concept_list]
results['greedy_selection'] = {
    'concepts': greedy_concept_list, 
    'values': greedy_reward,
}

On iteration 0
On iteration 1
On iteration 2
On iteration 3
On iteration 4
On iteration 5
On iteration 6
On iteration 7
On iteration 8
On iteration 9
On iteration 10
On iteration 11
On iteration 12
On iteration 13
On iteration 14
On iteration 15
On iteration 16
On iteration 17
On iteration 18
On iteration 19


In [36]:
manually_selected_concepts = [
    1,
    4,
    6,
    7,
    10,
    14,
    15,
    20,
    21,
    23,
    25,
    29,
    30,
    35,
    36,
    38,
    40,
    44,
    45,
    50,
    51,
    53,
    54,
    56,
    57,
    59,
    63,
    64,
    69,
    70,
    72,
    75,
    80,
    84,
    90,
    91,
    93,
    99,
    101,
    106,
    110,
    111,
    116,
    117,
    119,
    125,
    126,
    131,
    132,
    134,
    145,
    149,
    151,
    152,
    153,
    157,
    158,
    163,
    164,
    168,
    172,
    178,
    179,
    181,
    183,
    187,
    188,
    193,
    194,
    196,
    198,
    202,
    203,
    208,
    209,
    211,
    212,
    213,
    218,
    220,
    221,
    225,
    235,
    236,
    238,
    239,
    240,
    242,
    243,
    244,
    249,
    253,
    254,
    259,
    260,
    262,
    268,
    274,
    277,
    283,
    289,
    292,
    293,
    294,
    298,
    299,
    304,
    305,
    308,
    309,
    310,
    311,
]
results['manually_selected_concepts'] = manually_selected_concepts

In [37]:
random_from_manual_list = [np.random.choice(manually_selected_concepts,k,replace=False) for k in range(1,num_concepts_selected)]
random_from_manual_reward = [get_performance(c,np.ones(312)) for c in random_concept_list]

results['random_manual_selection'] = {
    'concepts': random_from_manual_list, 
    'values': random_from_manual_reward,
}

In [35]:
results['attribute_names'] = open("../../data/cub/attributes.txt").read().split("\n")

In [42]:
results['random_selection']['values'], results['greedy_selection']['values'], results['random_manual_selection']['values']

([0.007939247497411116,
  0.009147393855712806,
  0.018812564722126338,
  0.02105626510182948,
  0.018467380048325856,
  0.02226441146013117,
  0.026751812219537454,
  0.033310321021746636,
  0.02433551950293407,
  0.040904383845357266,
  0.03779772178115292,
  0.040386606834656544,
  0.06075250258888505,
  0.03503624439074905,
  0.07266137383500172,
  0.05919917155678288,
  0.045219192267863306,
  0.05246807041767346,
  0.1020020711080428],
 [0.005005177770107007,
  0.018639972385226095,
  0.032274767000345185,
  0.04573696927856403,
  0.06040731791508457,
  0.08025543665861236,
  0.09941318605453918,
  0.11166724197445634,
  0.13168795305488437,
  0.14325163962720056,
  0.1617190196755264,
  0.1667241974456334,
  0.17155678287884019,
  0.16465308940283052,
  0.1670693821194339,
  0.1782878840179496,
  0.18260269244045566,
  0.1886434242319641,
  0.20503969623748705,
  0.20331377286848465],
 [0.007939247497411116,
  0.009147393855712806,
  0.018812564722126338,
  0.02105626510182948,


## Save Data

In [ ]:
save_path = get_save_path(out_folder,save_name)

In [ ]:
delete_duplicate_results(out_folder,"",results)

In [ ]:
json.dump(results,open('../../results/'+save_path,'w'))